# 東証上場銘柄の証券コードを取得
https://www.jpx.co.jp/markets/statistics-equities/misc/01.html

In [1]:
import pandas as pd
df = pd.read_excel("../data/data_j.xls")
df.head(10)

/tmp/ipykernel_1519/2566161043.py:2: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  df = pd.read_excel("../data/data_j.xls")


,日付,コード,銘柄名,市場・商品区分,33業種コード,33業種区分,17業種コード,17業種区分,規模コード,規模区分
0,20251231,1301,極洋,プライム（内国株式）,50,水産・農林業,1,食品,6,TOPIX Small 1
1,20251231,1305,ｉＦｒｅｅＥＴＦ ＴＯＰＩＸ（年１回決算型）,ETF・ETN,-,-,-,-,-,-
2,20251231,1306,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ連動型上場投信,ETF・ETN,-,-,-,-,-,-
3,20251231,1308,上場インデックスファンドＴＯＰＩＸ,ETF・ETN,-,-,-,-,-,-
4,20251231,1309,ＮＥＸＴ ＦＵＮＤＳ ＣｈｉｎａＡＭＣ・中国株式・上証５０連動型上場投信,ETF・ETN,-,-,-,-,-,-
5,20251231,130A,Ｖｅｒｉｔａｓ Ｉｎ Ｓｉｌｉｃｏ,グロース（内国株式）,3250,医薬品,5,医薬品,-,-
6,20251231,1311,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ Ｃｏｒｅ ３０連動型上場投信,ETF・ETN,-,-,-,-,-,-
7,20251231,1319,ＮＥＸＴ ＦＵＮＤＳ 日経３００株価指数連動型上場投信,ETF・ETN,-,-,-,-,-,-
8,20251231,131A,ＣＣＮグループ,PRO Market,5250,情報・通信業,10,情報通信・サービスその他,-,-
9,20251231,1320,ｉＦｒｅｅＥＴＦ 日経２２５（年１回決算型）,ETF・ETN,-,-,-,-,-,-


In [2]:
# df のカラムとサンプルを確認して、銘柄名カラムを探す
print('columns:', df.columns.tolist())
df.head(3)

columns: ['日付', 'コード', '銘柄名', '市場・商品区分', '33業種コード', '33業種区分', '17業種コード', '17業種区分', '規模コード', '規模区分']


,日付,コード,銘柄名,市場・商品区分,33業種コード,33業種区分,17業種コード,17業種区分,規模コード,規模区分
0,20251231,1301,極洋,プライム（内国株式）,50,水産・農林業,1,食品,6,TOPIX Small 1
1,20251231,1305,ｉＦｒｅｅＥＴＦ ＴＯＰＩＸ（年１回決算型）,ETF・ETN,-,-,-,-,-,-
2,20251231,1306,ＮＥＸＴ ＦＵＮＤＳ ＴＯＰＩＸ連動型上場投信,ETF・ETN,-,-,-,-,-,-


証券コードのリストを取得

In [3]:
ticker_list = df["コード"].to_list()
ticker_list = [str(code) + ".T" for code in ticker_list]
print("日本株ティッカー例:", ticker_list[:5])

日本株ティッカー例: ['1301.T', '1305.T', '1306.T', '1308.T', '1309.T']


直近5年の株価を取得

In [4]:
# 日本株の価格データを小分けに取得し、失敗分を待機後に再試行する
import time
import pandas as pd
import yfinance as yf
import datetime as dt

today = dt.date.today()
start_date = today - dt.timedelta(days=365 * 5)  # 過去5年分

batch_size = 50
batch_pause_seconds = 2
retry_wait_seconds = [30, 60]
tickers = list(dict.fromkeys(ticker_list))
ticker_data = {}


def download_batch(batch):
    batch_df = yf.download(
        batch,
        start=start_date,
        end=today,
        group_by="ticker",
        threads=False,
        progress=False,
    )
    if batch_df.empty:
        return

    for ticker in batch:
        if isinstance(batch_df.columns, pd.MultiIndex):
            if ticker in batch_df.columns.get_level_values(0):
                ticker_df = batch_df[ticker]
            elif ticker in batch_df.columns.get_level_values(1):
                ticker_df = batch_df.xs(ticker, axis=1, level=1)
            else:
                continue
        elif len(batch) == 1:
            ticker_df = batch_df
        else:
            continue

        if "Close" in ticker_df.columns and ticker_df["Close"].notna().any():
            ticker_data[ticker] = ticker_df


def download_batches(batch_tickers):
    for start in range(0, len(batch_tickers), batch_size):
        batch = batch_tickers[start:start + batch_size]
        download_batch(batch)
        if start + batch_size < len(batch_tickers):
            time.sleep(batch_pause_seconds)


download_batches(tickers)

for retry_number, wait_seconds in enumerate(retry_wait_seconds, start=1):
    failed_tickers = [ticker for ticker in tickers if ticker not in ticker_data]
    if not failed_tickers:
        break
    print(f"未取得 {len(failed_tickers)} 銘柄。{wait_seconds} 秒待って再試行します ({retry_number}/{len(retry_wait_seconds)})。")
    time.sleep(wait_seconds)
    download_batches(failed_tickers)

failed_tickers = [ticker for ticker in tickers if ticker not in ticker_data]
stock_df = pd.concat(ticker_data, axis=1) if ticker_data else pd.DataFrame()

print(f"日本株ダウンロード完了。成功: {len(ticker_data)}/{len(tickers)} 銘柄、失敗: {len(failed_tickers)} 銘柄")
if failed_tickers:
    print("失敗銘柄の先頭30件:", failed_tickers[:30])

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 136A.T"}}}
$136A.T: possibly delisted; no timezone found

1 Failed download:
['136A.T']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 1432.T"}}}
$1432.T: possibly delisted; no timezone found
$1452.T: possibly delisted; no timezone found
$1445.T: possibly delisted; no timezone found

3 Failed downloads:
['1432.T', '1452.T', '1445.T']: possibly delisted; no timezone found
$1697.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1688.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1685.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1696.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1672.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$172

未取得 206 銘柄。30 秒待って再試行します (1/2)。


$2994.T: possibly delisted; no timezone found
$1776.T: possibly delisted; no timezone found
$1909.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2040.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1697.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2031.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2048.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2037.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2540.T: possibly delisted; no timezone found
$1432.T: possibly delisted; no timezone found
$1687.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2246.T: possibly delisted; no timezone found
$1452.T: possibly delisted; no timezone found
$1688.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2162.T: possibly delisted; no timezone found
$1685.T: possibly delisted; no price data foun

未取得 206 銘柄。60 秒待って再試行します (2/2)。


$2994.T: possibly delisted; no timezone found
$1776.T: possibly delisted; no timezone found
$1909.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2040.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$1697.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2031.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2048.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2037.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2540.T: possibly delisted; no timezone found
$1432.T: possibly delisted; no timezone found
$1687.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2246.T: possibly delisted; no timezone found
$1452.T: possibly delisted; no timezone found
$1688.T: possibly delisted; no price data found  (1d 2021-11-12 -> 2026-11-11)
$2162.T: possibly delisted; no timezone found
$1685.T: possibly delisted; no price data foun

日本株ダウンロード完了。成功: 4231/4437 銘柄、失敗: 206 銘柄
失敗銘柄の先頭30件: ['136A.T', '1432.T', '1445.T', '1452.T', '1672.T', '1685.T', '1687.T', '1688.T', '1696.T', '1697.T', '1726.T', '176A.T', '1776.T', '1841.T', '1909.T', '191A.T', '1948.T', '1949.T', '202A.T', '2031.T', '2032.T', '2033.T', '2036.T', '2037.T', '2040.T', '2041.T', '2043.T', '2048.T', '2065.T', '2066.T']


In [5]:
# 各ティッカーごとの最安値・最高値・現在値・0-100スケール値を作成
import pandas as pd

def compute_ticker_stats(stock_df):
    rows = []
    # マルチインデックス（ticker, OHLC）を想定
    if getattr(stock_df.columns, "nlevels", 1) > 1:
        tickers = sorted({c[0] for c in stock_df.columns})
        for t in tickers:
            # 安全に列を参照する（存在しない場合はスキップ）
            try:
                high_series = stock_df[(t, "High")]
                low_series = stock_df[(t, "Low")]
                close_series = stock_df[(t, "Close")].dropna()
            except Exception:
                # 別名の可能性に備え、柔軟に探す
                def find(col_names):
                    for name in col_names:
                        if (t, name) in stock_df:
                            return stock_df[(t, name)]
                    return None
                high_series = find(["High", "high", "HIGH"])
                low_series = find(["Low", "low", "LOW"])
                close_series = find(["Close", "close", "Adj Close", "Adj_Close"])  
                if close_series is not None:
                    close_series = close_series.dropna()
            if high_series is None or low_series is None or (close_series is None or close_series.empty):
                continue
            high = high_series.max()
            low = low_series.min()
            current = close_series.iloc[-1]
            scaled = (current - low) / (high - low) * 100 if pd.notna(high) and pd.notna(low) and high > low else float('nan')
            rows.append({"ticker": t, "最安値": low, "最高値": high, "現在値": current, "スケール(0-100)": scaled})
    else:
        # 単一レベル列の場合の簡易処理（必要に応じて拡張）
        cols = list(stock_df.columns)
        # ティッカーを推定
        tickers = sorted(set([str(c).split()[0].split('_')[0] for c in cols]))
        for t in tickers:
            def find_col(keyword):
                for c in cols:
                    s = str(c)
                    if s.startswith(t) and keyword in s:
                        return stock_df[c]
                return None
            high_series = find_col('High') or find_col('high')
            low_series = find_col('Low') or find_col('low')
            close_series = find_col('Close') or find_col('close')
            if high_series is None or low_series is None or close_series is None:
                continue
            high = high_series.max()
            low = low_series.min()
            close_series = close_series.dropna()
            if close_series.empty:
                continue
            current = close_series.iloc[-1]
            scaled = (current - low) / (high - low) * 100 if pd.notna(high) and pd.notna(low) and high > low else float('nan')
            rows.append({"ticker": t, "最安値": low, "最高値": high, "現在値": current, "スケール(0-100)": scaled})

    if not rows:
        return pd.DataFrame(
            columns=["最安値", "最高値", "現在値", "スケール(0-100)"],
            index=pd.Index([], name="ticker"),
        )
    result = pd.DataFrame(rows).set_index('ticker')
    return result

stats_df = compute_ticker_stats(stock_df)
# 見やすく順位付け（任意）
stats_df = stats_df.sort_values('スケール(0-100)', ascending=False)
stats_df

,最安値,最高値,現在値,スケール(0-100)
ticker,,,,
131A.T,1697.230225,1737.0,1737.0,100.0
333A.T,3204.586914,6240.0,6240.0,100.0
252A.T,2543.827393,2566.0,2566.0,100.0
374A.T,236.000000,240.0,240.0,100.0
279A.T,1530.000000,1550.0,1550.0,100.0
...,...,...,...,...
484A.T,430.000000,430.0,430.0,NaN
5622.T,1200.000000,1200.0,1200.0,NaN
5866.T,300.000000,300.0,300.0,NaN


In [6]:
# `stats_df` に `銘柄名` と `市場・商品区分` カラムを追加
# df の 'コード' を文字列にして辞書を作成
code_name = df.drop_duplicates('コード').set_index(df['コード'].astype(str))['銘柄名']
# stats_df のインデックスは例: '1711.T' なので '.T' を取り除く
codes = stats_df.index.to_series().apply(lambda x: str(x).split('.')[0])
stats_df['銘柄名'] = codes.map(code_name)
market_col = '市場・商品区分'
# df に市場カラムがあればマッピング、なければ None を埋める
if market_col in df.columns:
    code_market = df.drop_duplicates('コード').set_index(df['コード'].astype(str))[market_col]
    stats_df[market_col] = codes.map(code_market)
else:
    stats_df[market_col] = None
# 列の順序を調整（銘柄名と市場を先頭に）
cols = ['銘柄名', market_col] + [c for c in stats_df.columns if c not in ('銘柄名', market_col)]
stats_df = stats_df[cols]

# 表示
stats_df.head()

,銘柄名,市場・商品区分,最安値,最高値,現在値,スケール(0-100)
ticker,,,,,,
131A.T,ＣＣＮグループ,PRO Market,1697.230225,1737.0,1737.0,100.0
333A.T,テクノスマイル,PRO Market,3204.586914,6240.0,6240.0,100.0
252A.T,ウェッジ,PRO Market,2543.827393,2566.0,2566.0,100.0
374A.T,樋口総合研究所,PRO Market,236.000000,240.0,240.0,100.0
279A.T,インターグ,PRO Market,1530.000000,1550.0,1550.0,100.0


In [7]:
# スケールを昇順で表示（スクロール可能に）
from IPython.display import HTML, display

# 昇順にソートして数値を丸める
df_sorted = stats_df.sort_values('スケール(0-100)', ascending=True).copy()
df_sorted[['最安値', '最高値', '現在値', 'スケール(0-100)']] = df_sorted[['最安値', '最高値', '現在値', 'スケール(0-100)']].round(2)

# スクロール可能なコンテナで表示
html = f"""
<div style="max-height:600px; overflow:auto; border:1px solid #ddd; padding:8px;">
{df_sorted.to_html()}
</div>
"""

display(HTML(html))

,銘柄名,市場・商品区分,最安値,最高値,現在値,スケール(0-100)
ticker,,,,,,
5840.T,日本総険,PRO Market,5.050000e+02,1.010000e+03,5.050000e+02,0.00
248A.T,キッズスター,グロース（内国株式）,7.810000e+02,2.430000e+03,7.810000e+02,0.00
2990.T,アイダ設計,PRO Market,2.620000e+02,3.486800e+02,2.620000e+02,0.00
3174.T,ハピネス・アンド・ディ,スタンダード（内国株式）,3.010000e+02,9.135900e+02,3.010000e+02,0.00
9285.T,東京インフラ・エネルギー投資法人,REIT・ベンチャーファンド・カントリーファンド・インフラファンド,3.245000e+04,7.294587e+04,3.245000e+04,0.00
3557.T,ユナイテッド＆コレクティブ,グロース（内国株式）,4.260000e+02,1.319000e+03,4.260000e+02,0.00
8894.T,ＲＥＶＯＬＵＴＩＯＮ,スタンダード（内国株式）,1.300000e+01,6.910000e+02,1.300000e+01,0.00
326A.T,ａｅｒｏ ｌａｂ ｉｎｔｅｒｎａｔｉｏｎａｌ,PRO Market,1.620000e+03,1.620000e+04,1.620000e+03,0.00
5598.T,Ｙｏｔｔａｖｉａｓ,PRO Market,3.200000e+03,3.220000e+03,3.200000e+03,0.00


In [8]:
print("a")

a
